# DS-02 — National Public Toilet Map

DS-02 supplies the *toilet* link of the access chain, and it is the richest
accessibility source in the register: it publishes accessible, ambulant, transfer
side, MLAK key requirement, adult change and Changing Places as separate structured
fields rather than as free text.

Two things to settle before the column contract:

1. **Extent.** The file is national. What is the Victorian and Greater Melbourne subset?
2. **Attribute semantics.** These are boolean columns. Is a false a published *no*, or
   is it an unfilled default? That distinction decides whether the venue card can ever
   say "no accessible toilet" or only "no published information".

In [1]:
import sys, warnings
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
warnings.filterwarnings("ignore")

import pandas as pd
import profile_lib as pl

pd.set_option("display.max_rows", 120)
pd.set_option("display.width", 200)

print("project root:", pl.PROJECT_ROOT)
print("raw zone:    ", pl.RAW_ROOT)


project root: C:\Users\nitin\Documents\Projects\Final_Project\SportAble
raw zone:     C:\Users\nitin\Documents\Projects\Final_Project\SportAble\_raw


In [2]:
raw = pl.resolve("DS-02")
p = pl.Profile(raw, "National Public Toilet Map")
p.check("raw_integrity",
        "pass" if raw.sha_matches_manifest else ("info" if raw.sha_matches_manifest is None else "fail"),
        f"SHA-256 of the profiled object is {raw.sha256}", raw.sha256)
raw

RawObject(DS-02 dt=2026-08-31 toiletmapexport_260801_074429.csv 11,995,188B sha=8aaef33e53f0… [match])

In [3]:
df = pd.read_csv(raw.path, low_memory=False)
p.observe("row_count_national", int(len(df)))
p.observe("column_count", int(len(df.columns)))
print(f"{len(df):,} rows x {len(df.columns)} columns")
list(df.columns)

25,449 rows x 47 columns


['FacilityID',
 'URL',
 'Name',
 'FacilityType',
 'Address1',
 'Town',
 'State',
 'AddressNote',
 'Latitude',
 'Longitude',
 'Parking',
 'ParkingAccessible',
 'ParkingNote',
 'KeyRequired',
 'MLAK24',
 'MLAKAfterHours',
 'PaymentRequired',
 'AccessNote',
 'AdultChange',
 'ChangingPlaces',
 'BYOSling',
 'ACShower',
 'ACMLAK',
 'AdultChangeNote',
 'BabyChange',
 'BabyCareRoom',
 'BabyChangeNote',
 'DumpPoint',
 'DPWashout',
 'DPAfterHours',
 'DumpPointNote',
 'OpeningHours',
 'OpeningHoursNote',
 'Male',
 'Female',
 'Unisex',
 'AllGender',
 'Ambulant',
 'Accessible',
 'LHTransfer',
 'RHTransfer',
 'ToiletNote',
 'SharpsDisposal',
 'DrinkingWater',
 'SanitaryDisposal',
 'MensPadDisposal',
 'Shower']

## 1. Extent

In [4]:
by_state = df["State"].value_counts(dropna=False)
p.observe("rows_by_state", by_state)
vic = df[df["State"].eq("VIC")].copy()
p.observe("row_count_vic", int(len(vic)))
p.check("extent_vic", "info",
        f"{len(vic):,} of {len(df):,} national rows are Victorian", int(len(vic)))
by_state

State
NSW    7912
VIC    6059
QLD    4751
WA     2887
SA     2165
TAS    1007
ACT     341
NT      325
NaN       2
Name: count, dtype: int64

In [5]:
coord_stats = pl.check_coordinates(p, vic, "Latitude", "Longitude", label="VIC rows")
gm_box = vic[
    vic["Latitude"].between(pl.GM_BBOX["min_lat"], pl.GM_BBOX["max_lat"])
    & vic["Longitude"].between(pl.GM_BBOX["min_lon"], pl.GM_BBOX["max_lon"])
].copy()
p.observe("row_count_gm_bbox", int(len(gm_box)))
p.contract(f"Clip DS-02 spatially against {pl.LGA_BOUNDARY_ID} rather than filtering on State or Town.")
coord_stats

{'total': 6059,
 'null_coords': 0,
 'out_of_range': 0,
 'likely_swapped_lat_lon': 0,
 'inside_gm_bbox': 3373,
 'outside_gm_bbox': 2686,
 'pct_inside_gm_bbox': 55.67}

In [6]:
dupes = int(df["FacilityID"].duplicated().sum())
p.observe("duplicate_facility_id", dupes)
p.check("primary_key", "pass" if dupes == 0 else "fail",
        f"FacilityID is {'unique' if dupes == 0 else f'NOT unique — {dupes:,} duplicates'}", dupes)
p.contract("FacilityID is the natural key for DS-02 and is enforced unique on load.")
dupes

0

## 2. Accessibility attributes

Counted on the Greater Melbourne bounding box subset, which is the population the
product actually serves.

In [7]:
ACCESS_FIELDS = [
    "Accessible", "Ambulant", "LHTransfer", "RHTransfer",
    "ChangingPlaces", "AdultChange", "ACShower", "ACMLAK", "BYOSling",
    "KeyRequired", "MLAK24", "MLAKAfterHours",
    "ParkingAccessible", "Parking", "PaymentRequired",
    "BabyChange", "BabyCareRoom", "Shower", "DrinkingWater",
]
rowsets = {"vic": vic, "greater_melbourne_bbox": gm_box}
summary = {}
for label, frame in rowsets.items():
    for col in ACCESS_FIELDS:
        s = frame[col]
        summary.setdefault(col, {})[f"{label}_true"] = int((s == True).sum())
        summary.setdefault(col, {})[f"{label}_false"] = int((s == False).sum())
        summary.setdefault(col, {})[f"{label}_null"] = int(s.isna().sum())
acc = pd.DataFrame(summary).T
p.observe("accessibility_field_counts", acc)
acc

,vic_true,vic_false,vic_null,greater_melbourne_bbox_true,greater_melbourne_bbox_false,greater_melbourne_bbox_null
Accessible,3475,2584,0,2238,1135,0
Ambulant,615,5444,0,456,2917,0
LHTransfer,371,5688,0,252,3121,0
RHTransfer,371,5688,0,265,3108,0
ChangingPlaces,163,5896,0,123,3250,0
AdultChange,243,5816,0,185,3188,0
ACShower,104,5955,0,75,3298,0
ACMLAK,0,6059,0,0,3373,0
BYOSling,213,5846,0,160,3213,0
KeyRequired,108,5951,0,54,3319,0


In [8]:
# Does false mean a published no, or an unfilled default?
never_null = [c for c in ACCESS_FIELDS if gm_box[c].isna().sum() == 0]
sometimes_null = [c for c in ACCESS_FIELDS if gm_box[c].isna().sum() > 0]
p.observe("boolean_fields_never_null", never_null)
p.observe("boolean_fields_sometimes_null", sometimes_null)
p.check(
    "boolean_semantics",
    "warn" if never_null else "pass",
    (f"{len(never_null)} accessibility booleans are never null in the Greater Melbourne subset, so "
     "the publisher writes false rather than leaving blank — false cannot be read as a surveyed 'no', "
     "only as 'not recorded as present'"
     if never_null else "accessibility booleans carry nulls, so false is distinguishable from unfilled"),
    {"never_null": len(never_null), "sometimes_null": len(sometimes_null)},
)
p.contract("Map DS-02 booleans to tri-state: true -> PUBLISHED_YES, false -> NOT_RECORDED, null -> NOT_RECORDED. Never render a DS-02 false as 'no accessible toilet'.")
p.limitation(
    "DS-02 publishes accessibility attributes as booleans with no separate unknown value. "
    "A false is treated as no published information rather than as a surveyed absence."
)
{"never_null": never_null, "sometimes_null": sometimes_null}

{'never_null': ['Accessible',
  'Ambulant',
  'LHTransfer',
  'RHTransfer',
  'ChangingPlaces',
  'AdultChange',
  'ACShower',
  'ACMLAK',
  'BYOSling',
  'KeyRequired',
  'MLAK24',
  'MLAKAfterHours',
  'ParkingAccessible',
  'Parking',
  'PaymentRequired',
  'BabyChange',
  'BabyCareRoom',
  'Shower',
  'DrinkingWater'],
 'sometimes_null': []}

In [9]:
# The subset that actually helps a wheelchair user, and the conditions attached to it.
acc_gm = gm_box[gm_box["Accessible"] == True]
gated = {
    "accessible_total": int(len(acc_gm)),
    "accessible_key_required": int((acc_gm["KeyRequired"] == True).sum()),
    "accessible_mlak_24h": int((acc_gm["MLAK24"] == True).sum()),
    "accessible_payment_required": int((acc_gm["PaymentRequired"] == True).sum()),
    "accessible_with_accessible_parking": int((acc_gm["ParkingAccessible"] == True).sum()),
    "accessible_left_transfer": int((acc_gm["LHTransfer"] == True).sum()),
    "accessible_right_transfer": int((acc_gm["RHTransfer"] == True).sum()),
    "accessible_transfer_side_unstated": int(((acc_gm["LHTransfer"] != True) & (acc_gm["RHTransfer"] != True)).sum()),
    "accessible_opening_hours_stated": int(acc_gm["OpeningHours"].notna().sum()),
}
p.observe("accessible_toilet_conditions_gm", gated)
p.check("access_conditions", "info",
        f"of {gated['accessible_total']:,} accessible toilets in the Greater Melbourne bounding box, "
        f"{gated['accessible_key_required']:,} require a key and {gated['accessible_payment_required']:,} require payment — "
        "an accessible toilet behind an MLAK key is not the same answer as an open one",
        gated)
p.contract("Carry KeyRequired, MLAK24 and PaymentRequired through to the venue card. An accessible toilet with a key requirement is displayed with that condition, not as a plain yes.")
pd.Series(gated)

accessible_total                      2238
accessible_key_required                 28
accessible_mlak_24h                     89
accessible_payment_required              9
accessible_with_accessible_parking     970
accessible_left_transfer               250
accessible_right_transfer              264
accessible_transfer_side_unstated     1841
accessible_opening_hours_stated       2238
dtype: int64

In [10]:
staleness = {
    "opening_hours_null": int(gm_box["OpeningHours"].isna().sum()),
    "access_note_present": int(gm_box["AccessNote"].notna().sum()),
    "toilet_note_present": int(gm_box["ToiletNote"].notna().sum()),
}
p.observe("free_text_fields", staleness,
          "free text notes are displayed verbatim with attribution, never parsed for attributes")
p.contract("AccessNote and ToiletNote are displayed verbatim. No attribute is inferred from them.")
pd.Series(staleness)

opening_hours_null       0
access_note_present    138
toilet_note_present    441
dtype: int64

In [11]:
p.save()

DS-02 — National Public Toilet Map
  object   toiletmapexport_260801_074429.csv  (11,995,188 bytes)
  dt       2026-08-31
  sha256   8aaef33e53f0e8ed918d2782f9a4bedb9fc963b90e36139a8892ea55c4056fea
  manifest hash matches

  Checks (WARN overall)
    [PASS] raw_integrity: SHA-256 of the profiled object is 8aaef33e53f0e8ed918d2782f9a4bedb9fc963b90e36139a8892ea55c4056fea
    [INFO] extent_vic: 6,059 of 25,449 national rows are Victorian
    [PASS] coords_present: 0 of 6,059 VIC rows have a null coordinate
    [PASS] coords_in_range: 0 VIC rows have a coordinate outside valid lat/lon range
    [PASS] coords_not_transposed: no VIC rows have latitude and longitude in the wrong columns
    [INFO] coords_greater_melbourne: 3,373 of 6,059 VIC rows fall inside the Greater Melbourne bounding box (55.67%) — coarse screen only, the clip is spatial against DS-06
    [PASS] primary_key: FacilityID is unique
    [WARN] boolean_semantics: 19 accessibility booleans are never null in the Greater Melbour

WindowsPath('C:/Users/nitin/Documents/Projects/Final_Project/SportAble/_profiles/dt=2026-08-31/DS-02.json')